# Generación de Figuras (Paneles) desde Cero

Este notebook contiene el código línea por línea para generar cada una de las figuras sin depender de llamar a las funciones del script externo.


In [ ]:
%matplotlib inline
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Importar funciones base y datos de los scripts del repositorio
import analisis_common as A
import analisis_figuras_jfm as J
%matplotlib inline
from analisis_figuras_jfm import (M, PHIS, CMAP_PHI0, A_DIFF, DUNE, SL,
                                  _load, perfil_x_medio, ell_faces,
                                  perfil_equilibrio, indice_gradacion,
                                  numeros_adim)

# Configuración inicial y constantes
ETA_A = float(np.mean(A.eta_active_layer(M)[DUNE]))

def _ejes(ax):
    ax.tick_params(direction='in', top=True, right=True, labelsize=9)



## Figura 1: Perfil de sorting vertical


In [ ]:
# F1_a
fig, ax = plt.subplots(figsize=(6.4, 5.6))
fig.patch.set_facecolor('white')

for k, p0 in enumerate(PHIS):
    ph, _ = _load('dif', p0)
    pr = perfil_x_medio(ph)
    ax.plot(pr, M.ec, color=CMAP_PHI0(k / (len(PHIS) - 1)), lw=2.0,
            label=f"$\\phi_s^0$={p0:.1f}")

phf = A._phi_faces(M, _load('dif', 0.7)[0])
ell_a = float(np.median(ell_faces(phf)[DUNE][:, -6:]))

for k, p0 in enumerate(PHIS):
    peq, eeq = perfil_equilibrio(p0, ell_a, ETA_A, 1.0)
    ax.plot(peq, eeq, color=CMAP_PHI0(k / (len(PHIS) - 1)), lw=1.1, ls='--')

ax.plot([], [], color='0.35', lw=1.1, ls='--', label='Gray & Chugunov (2006)\nequilibrium')
ax.axhline(ETA_A, color='0.35', ls=':', lw=1.3)
ax.text(0.02, ETA_A + 0.015, r'$\langle\eta_a\rangle$', fontsize=9, color='0.3')
ax.set_xlim(-0.02, 1.02); ax.set_ylim(0, 1)
ax.set_xlabel(r'$\langle\phi_s\rangle_x$', fontsize=11)
ax.set_ylabel(r'$\eta = z/h$', fontsize=11)
ax.legend(fontsize=8.5, loc='center left', framealpha=0.92)
_ejes(ax)
plt.title('(a) with diffusion ($A$=%.3f), $i$=0.00975' % A_DIFF, fontsize=11, loc='left')
plt.show()


In [ ]:
# F1_b
fig, ax = plt.subplots(figsize=(6.4, 5.6))
fig.patch.set_facecolor('white')

for k, p0 in enumerate(PHIS):
    ph, _ = _load('hyp', p0)
    pr = perfil_x_medio(ph)
    ax.plot(pr, M.ec, color=CMAP_PHI0(k / (len(PHIS) - 1)), lw=2.0,
            label=f"$\\phi_s^0$={p0:.1f}")

ax.axhline(ETA_A, color='0.35', ls=':', lw=1.3)
ax.text(0.02, ETA_A + 0.015, r'$\langle\eta_a\rangle$', fontsize=9, color='0.3')
ax.set_xlim(-0.02, 1.02); ax.set_ylim(0, 1)
ax.set_xlabel(r'$\langle\phi_s\rangle_x$', fontsize=11)
ax.set_ylabel(r'$\eta = z/h$', fontsize=11)
ax.legend(fontsize=8.5, loc='center left', framealpha=0.92)
_ejes(ax)
plt.title('(b) no diffusion (purely hyperbolic)', fontsize=11, loc='left')
plt.show()


In [ ]:
# F1_c
fig, ax = plt.subplots(figsize=(6.4, 5.6))
fig.patch.set_facecolor('white')
win = M.ec < ETA_A
lo = win & (M.ec < 0.5 * ETA_A)
hi = win & (M.ec >= 0.5 * ETA_A)

for kind, c, mk, lab in (('dif', '#c62828', 'o', 'with diffusion'),
                         ('hyp', '#1565c0', 's', 'no diffusion')):
    idx = []
    for p in PHIS:
        pr = perfil_x_medio(_load(kind, p)[0])
        idx.append(pr[hi].mean() - pr[lo].mean())
    
    ax.plot(PHIS, idx, mk + '-', color=c, lw=1.8, ms=7, label=lab)
    for p, v in zip(PHIS, idx):
        ax.annotate(f'{v:+.2f}', (p, v), textcoords='offset points',
                    xytext=(-17 if kind == 'dif' else 17, -3),
                    ha='center', fontsize=7.5, color=c)

ax.axhline(0.0, color='k', lw=1.0)
ax.set_xlabel(r'$\phi_s^0$', fontsize=11)
ax.set_ylabel(r'$\langle\phi_s\rangle_{\rm upper} - \langle\phi_s\rangle_{\rm lower}$', fontsize=11)
ax.legend(fontsize=9, loc='best', framealpha=0.92)
_ejes(ax)
plt.show()


## Figura 2: Zona de gruesos atrapada


In [ ]:
# Cargar datos para figura 2
d_f2 = np.load(os.path.join(J.OD, 'Single_slope_model_cmp_timeseries.npz'))
ph_f2 = d_f2['phi_full'][-1]
t0_f2 = 0.25 * M.T_period


In [ ]:
# F2_a
xmm = M.xc[SL] * 1e3
X = np.repeat(xmm[:, None], M.Nz, axis=1)
Z = M.Ec[SL] * M.h2[SL] * 1e3
u_c = A.u_centers(M, t0_f2)

fig, ax = plt.subplots(figsize=(10.5, 4.4))
fig.patch.set_facecolor('white')
pm = ax.pcolormesh(X, Z, ph_f2[SL], cmap=A.CMAP_PHI, vmin=0, vmax=1,
                   shading='gouraud', rasterized=True)
zg, Ud, Wd = A._dune_frame_fields(M, t0_f2)
ax.streamplot(xmm, zg * 1e3, np.nan_to_num(Ud[:, SL]) * 1e3,
              np.nan_to_num(Wd[:, SL]) * 1e3, color='0.25',
              density=1.3, linewidth=0.7, arrowsize=0.7)
ax.fill_between(xmm, M.h[SL] * 1e3, zg.max() * 1e3 * 1.02, color='white', zorder=3, lw=0)
ax.contour(X, Z, u_c[SL] * 1e3, levels=[0.0], colors='#00e5ff', linewidths=2.6, zorder=4)
ax.plot(xmm, M.h[SL] * 1e3, 'k-', lw=1.8, zorder=5)
ax.plot(xmm, (M.h[SL] - M.delta_a) * 1e3, color='0.3', ls=':', lw=1.3, zorder=5)
ax.plot([], [], color='#00e5ff', lw=2.6, label=r'$u=0$')
ax.plot([], [], color='0.25', lw=0.9, label=r'streamlines of $(u,\,w_z)$')
ax.plot([], [], color='0.3', ls=':', lw=1.3, label=r'$\eta_a$')
ax.set_ylim(0, M.h.max() * 1e3 * 1.02)
ax.set_xlabel(r'$x$ (mm)', fontsize=11)
ax.set_ylabel(r'$z$ (mm)', fontsize=11)
ax.legend(fontsize=8.5, loc='upper right', framealpha=0.92)
ax.invert_xaxis()
_ejes(ax)
cb = fig.colorbar(pm, ax=ax, pad=0.012, fraction=0.030)
cb.set_label(r'$\phi_s$ — fines fraction', fontsize=10)
plt.show()


In [ ]:
# F2_b
xmm = M.xc[SL] * 1e3
u_c = A.u_centers(M, t0_f2)
w_c = A.w_eta_centers(M, t0_f2)

fig, ax = plt.subplots(figsize=(10.5, 4.4))
fig.patch.set_facecolor('white')
pm = ax.pcolormesh(xmm, M.ec, ph_f2[SL].T, cmap=A.CMAP_PHI, vmin=0, vmax=1,
                   shading='gouraud', rasterized=True)
ax.streamplot(xmm, M.ec, u_c[SL].T * 1e3, (w_c / M.h2)[SL].T, color='0.25',
              density=1.3, linewidth=0.7, arrowsize=0.7)
ax.contour(xmm, M.ec, u_c[SL].T * 1e3, levels=[0.0], colors='#00e5ff', linewidths=2.6)
e_a = A.eta_active_layer(M)
ax.plot(xmm, e_a[SL], color='0.3', ls=':', lw=1.3)

e_coarse = np.full(M.Nx, np.nan)
for i in range(M.Nx):
    m = M.ec < e_a[i] - 0.02
    if m.sum() > 3:
        e_coarse[i] = M.ec[m][int(np.argmin(ph_f2[i][m]))]
ax.plot(xmm, e_coarse[SL], color='#00e5ff', ls='none', marker='o', ms=2.6,
        mfc='none', mew=0.9, label=r'minimum of $\phi_s$ below $\eta_a$')

ax.plot([], [], color='#00e5ff', lw=2.6, label=r'$u=0$')
ax.plot([], [], color='0.3', ls=':', lw=1.3, label=r'$\eta_a$')
ax.set_xlabel(r'$x$ (mm)', fontsize=11)
ax.set_ylabel(r'$\eta=z/h$', fontsize=11)
ax.set_ylim(0, 1)
ax.legend(fontsize=8, loc='upper left', framealpha=0.92)
ax.invert_xaxis()
_ejes(ax)
cb = fig.colorbar(pm, ax=ax, pad=0.012, fraction=0.030)
cb.set_label(r'$\phi_s$ — fines fraction', fontsize=10)
plt.show()


## Figura 4: Campos de $\ell$ y $Pe$


In [ ]:
# Cargar datos para figura 4
ph_f4, _ = _load('dif', 0.7)
phf_f4 = A._phi_faces(M, ph_f4)
ell_f4 = ell_faces(phf_f4)
Pe_f4 = M.h2 / ell_f4


In [ ]:
# F4_a
xmm = M.xc[SL] * 1e3
ef = M.eta_f
fig, ax = plt.subplots(figsize=(8.2, 4.6))
fig.patch.set_facecolor('white')
pm = ax.pcolormesh(xmm, ef, ell_f4[SL].T * 1e3, cmap='magma', shading='gouraud',
                   rasterized=True)
cs = ax.contour(xmm, ef, ell_f4[SL].T * 1e3, levels=[0.1, 0.15, 0.2, 0.3],
                colors='w', linewidths=0.9)
ax.clabel(cs, fmt='%g', fontsize=8)
ax.plot(xmm, A.eta_active_layer(M)[SL], color='#00e5ff', ls=':', lw=1.6, label=r'$\eta_a$')
ax.set_xlabel(r'$x$ (mm)', fontsize=11)
ax.set_ylabel(r'$\eta=z/h$', fontsize=11)

ax.legend(fontsize=9, loc='lower left', framealpha=0.9)
_ejes(ax)
cb = fig.colorbar(pm, ax=ax, pad=0.012, fraction=0.045)
cb.set_label(r'$\ell$ (mm)', fontsize=10)
plt.show()


In [ ]:
# F4_b
xmm = M.xc[SL] * 1e3
ef = M.eta_f
fig, ax = plt.subplots(figsize=(8.2, 4.6))
fig.patch.set_facecolor('white')
pm = ax.pcolormesh(xmm, ef, np.log10(Pe_f4[SL]).T, cmap='cividis', shading='gouraud',
                   rasterized=True)
cs = ax.contour(xmm, ef, Pe_f4[SL].T, levels=[1, 3, 10, 30, 100], colors='w', linewidths=0.9)
ax.clabel(cs, fmt='%g', fontsize=8)
ax.plot(xmm, A.eta_active_layer(M)[SL], color='r', ls=':', lw=1.6, label=r'$\eta_a$')
ax.set_xlabel(r'$x$ (mm)', fontsize=11)
ax.set_ylabel(r'$\eta=z/h$', fontsize=11)

ax.legend(fontsize=9, loc='lower left', framealpha=0.9)
_ejes(ax)
cb = fig.colorbar(pm, ax=ax, pad=0.012, fraction=0.045)
cb.set_label(r'$\log_{10}Pe$', fontsize=10)
plt.show()


In [ ]:
# F4_c
STATION_EN = {'stoss': 'stoss (mid)', 'cresta': 'crest', 'lee25': 'lee 25%',
              'lee50': 'lee 50%', 'lee75': 'lee 75%', 'pie_lee': 'lee toe'}

Pe_da = M.delta_a / ell_f4
ef = M.eta_f

fig, ax = plt.subplots(figsize=(6.6, 5.6))
fig.patch.set_facecolor('white')
for key, label, ix, xv in A.stations(M):
    ax.semilogx(Pe_f4[ix], ef, lw=1.8, color=A.STATION_COLORS[key],
                label=STATION_EN.get(key, key))
    ax.semilogx(Pe_da[ix], ef, lw=1.0, ls='--', color=A.STATION_COLORS[key], alpha=0.6)
ax.axvline(1.0, color='k', lw=1.2)
ax.set_xlabel(r'$Pe$   (—— $L=h$ ;  - - - $L=\delta_a$)', fontsize=11)
ax.set_ylabel(r'$\eta=z/h$', fontsize=11); ax.set_ylim(0, 1)

ax.legend(fontsize=8.5, loc='lower left', framealpha=0.92)
ax.tick_params(direction='in', top=True, right=True, labelsize=9, which='both')
plt.show()


In [ ]:
# F4_d
ef = M.eta_f
ix = dict((s[0], s[2]) for s in A.stations(M))[A.STATION_REF]
p_lit = M.nu_pack * M.h[ix] * (1 - ef)

fig, ax = plt.subplots(figsize=(6.6, 5.6))
fig.patch.set_facecolor('white')
ax.plot(ell_f4[ix] * 1e3, ef, color='#c62828', lw=2.0,
        label=r'$\ell$ with $p=\nu h(1-\eta)+p_{floor}$')
ell_nofloor = (A_DIFF * (M.C_seg * ((1 - phf_f4[ix]) * M.d_l + phf_f4[ix] * M.d_s) + p_lit)
               / (M.B_seg * ((M.R - 1) + M.E_seg * (1 - phf_f4[ix]) * (M.R - 1)**2)))
ax.plot(ell_nofloor * 1e3, ef, color='#1565c0', lw=1.6, ls='--',
        label=r'$\ell$ without the $p_{floor}$ regularisation')
ax.axvline(M.h[ix] * 1e3, color='0.4', ls='-.', lw=1.2, label=r'local $h$')
ax.axvline(M.delta_a * 1e3, color='0.4', ls=':', lw=1.2, label=r'$\delta_a$')
ax.axhline(A.eta_active_layer(M)[ix], color='0.4', ls=':', lw=1.2)
ax.set_xscale('log')
ax.set_xlabel(r'$\ell$ (mm)', fontsize=11)
ax.set_ylabel(r'$\eta=z/h$', fontsize=11)
ax.set_ylim(0, 1)

ax.legend(fontsize=8.5, loc='upper left', framealpha=0.92)
ax.tick_params(direction='in', top=True, right=True, labelsize=9, which='both')
plt.show()


## Figura 6: Convergencia al estado de onda viajera


In [ ]:
# Cargar datos para figura 6
import Single_slope_model as MS
d_f6 = np.load(os.path.join(J.OD, 'Single_slope_model_timeseries.npz'))
t_full = d_f6['t_full']
phi_full = d_f6['phi_full']
i0 = int(np.searchsorted(MS.xc, MS.x_dune0))
i1 = int(np.searchsorted(MS.xc, MS.x_lee_toe))
dn = slice(i0, i1)


In [ ]:
# F6_a
L2 = []
for k, t in enumerate(t_full):
    k2 = int(np.argmin(np.abs(t_full - 2 * t)))
    if t == 0 or t_full[k2] < 2 * t * 0.98:
        L2.append(np.nan)
        continue
    a, b = phi_full[k][dn], phi_full[k2][dn]
    L2.append(np.linalg.norm(a - b) / np.linalg.norm(b))

L2 = np.array(L2)
fig, ax = plt.subplots(figsize=(6.6, 5.0))
fig.patch.set_facecolor('white')
ax.semilogy(t_full, L2, 'o-', color='#c62828', lw=1.6, ms=4,
            label=r'$\|\phi_s(t)-\phi_s(2t)\|_2 / \|\phi_s(2t)\|_2$')
ax.axvline(2400, color='0.6', lw=1.0)
ax.set_xlim(0, t_full[-1] / 2 * 1.05)
ax.set_xlabel(r'$t$ (s)', fontsize=11)
ax.set_ylabel('relative norm', fontsize=11)
ax.set_title('(a) convergence in the co-moving frame', fontsize=11, loc='left')
ax.legend(fontsize=8.5, loc='upper right', framealpha=0.92)
ax.tick_params(direction='in', top=True, right=True, labelsize=9, which='both')
plt.show()


In [ ]:
# F6_a1
e_a = A.eta_active_layer(MS)
eta_min = []
for k, t in enumerate(t_full):
    ph = phi_full[k]
    min_eta_list = []
    for i in range(dn.start, dn.stop):
        m = MS.ec < e_a[i] - 0.02
        if m.sum() > 3:
            min_eta_list.append(MS.ec[m][int(np.argmin(ph[i][m]))])
    if min_eta_list:
        eta_min.append(np.mean(min_eta_list))
    else:
        eta_min.append(np.nan)

fig, ax = plt.subplots(figsize=(6.6, 5.0))
fig.patch.set_facecolor('white')
ax.plot(t_full, eta_min, 'o-', color='#1565c0', lw=1.6, ms=4,
        label=r'mean $\eta$ of coarse core')
ax.axvline(2400, color='0.6', lw=1.0)
ax.set_xlim(0, t_full[-1] * 1.05)
ax.set_xlabel(r'$t$ (s)', fontsize=11)
ax.set_ylabel(r'$\eta$ position', fontsize=11)
ax.set_title('(a1) geometric tracking (coarse core depth)', fontsize=11, loc='left')
ax.legend(fontsize=8.5, loc='upper right', framealpha=0.92)
ax.tick_params(direction='in', top=True, right=True, labelsize=9, which='both')
plt.show()


In [ ]:
# F6_a2
rate = []
t_mid = []
for k in range(len(t_full)-1):
    dt = t_full[k+1] - t_full[k]
    if dt <= 0: continue
    a = phi_full[k+1][dn]
    b = phi_full[k][dn]
    val = np.linalg.norm(a - b) / dt
    rate.append(val)
    t_mid.append((t_full[k+1] + t_full[k]) / 2.0)

fig, ax = plt.subplots(figsize=(6.6, 5.0))
fig.patch.set_facecolor('white')
ax.semilogy(t_mid, rate, 'o-', color='#2e7d32', lw=1.6, ms=4,
            label=r'$\|\phi_s(t+\Delta t) - \phi_s(t)\|_2 / \Delta t$')
ax.axvline(2400, color='0.6', lw=1.0)
ax.set_xlim(0, t_full[-1] * 1.05)
ax.set_xlabel(r'$t$ (s)', fontsize=11)
ax.set_ylabel('rate of change (1/s)', fontsize=11)
ax.set_title('(a2) instantaneous rate of change', fontsize=11, loc='left')
ax.legend(fontsize=8.5, loc='upper right', framealpha=0.92)
ax.tick_params(direction='in', top=True, right=True, labelsize=9, which='both')
plt.show()


In [ ]:
# F6_b
w = MS.h[dn][:, None]
fig, ax = plt.subplots(figsize=(6.6, 5.6))
fig.patch.set_facecolor('white')
cmap = plt.get_cmap('viridis')
sel = np.linspace(0, len(t_full) - 1, 9).astype(int)
for m_, k in enumerate(sel):
    pr = (phi_full[k][dn] * w).sum(axis=0) / w.sum()
    es_ic = (k == 0)
    ax.plot(pr, MS.ec, lw=1.8 if es_ic else 1.6,
            ls=':' if es_ic else '-',
            color='0.25' if es_ic else cmap(m_ / (len(sel) - 1)),
            label=(f'$t$=0 s  ($\phi_s^0$={MS.PHI_S:.2f}, I.C.)' if es_ic
                   else f'$t$={t_full[k]:.0f} s'))
ax.set_xlim(-0.02, 1.02); ax.set_ylim(0, 1)
ax.set_xlabel(r'$\langle\phi_s\rangle_x$', fontsize=11)
ax.set_ylabel(r'$\eta=z/h$', fontsize=11)
ax.set_title('(b) collapse of the $x$-averaged sorting profile', fontsize=11, loc='left')
ax.legend(fontsize=8, loc='center left', ncol=2, framealpha=0.92)
_ejes(ax)
plt.show()


## Figura 8: Barrido en pendiente


In [ ]:
# F8_a
RUNS = os.path.join(J.OD, 'runs')
datos_f8 = []
for i_tag, i_val in (('i2', 0.00730), ('i1', 0.00975)):
    datos_f8.append((i_val, _load('dif', 0.7, i_tag)[0]))

for tag, i_val in (('F8_i0050', 0.0050), ('F8_i0140', 0.0140)):
    f = os.path.join(RUNS, f'{tag}_timeseries.npz')
    if os.path.exists(f):
        datos_f8.append((i_val, np.load(f)['phi_full'][-1]))

fig, ax = plt.subplots(figsize=(6.6, 5.6))
fig.patch.set_facecolor('white')
cmi = plt.get_cmap('inferno')
datos_f8.sort()

for k, (iv, ph) in enumerate(datos_f8):
    na = numeros_adim(0.7, iv)
    ax.plot(perfil_x_medio(ph), M.ec, lw=2.0,
            color=cmi(0.15 + 0.7 * k / max(len(datos_f8) - 1, 1)),
            label=f"$i$={iv:.5f}  ($\Lambda$={na['Lambda']:.0f})")
ax.axhline(ETA_A, color='0.4', ls=':', lw=1.2)
ax.set_xlim(-0.02, 1.02); ax.set_ylim(0, 1)
ax.set_xlabel(r'$\langle\phi_s\rangle_x$', fontsize=11)
ax.set_ylabel(r'$\eta=z/h$', fontsize=11)
ax.set_title(r'(a) effect of the slope at $\phi_s^0$=0.70', fontsize=11, loc='left')
ax.legend(fontsize=8.5, loc='center left', framealpha=0.92)
_ejes(ax)
plt.show()


In [ ]:
# F8_c
cmp_f = os.path.join(J.OD, 'Single_slope_model_cmp_timeseries.npz')
fuentes = [(0.0050, os.path.join(RUNS, 'F8_i0050_timeseries.npz')),
           (0.00975, cmp_f),
           (0.0140, os.path.join(RUNS, 'F8_i0140_timeseries.npz'))]
series = []
for iv, f in fuentes:
    if not os.path.exists(f):
        continue
    d = np.load(f)
    series.append((iv, d['t_full'], np.array([indice_gradacion(p_) for p_ in d['phi_full']])))
series.sort()

fig, ax = plt.subplots(figsize=(6.8, 5.4))
fig.patch.set_facecolor('white')

for k, (iv, t, g) in enumerate(series):
    na = numeros_adim(0.7, iv)
    ax.plot(t, g, 'o-', ms=3.5, lw=1.7,
            color=cmi(0.15 + 0.7 * k / max(len(series) - 1, 1)),
            label=f"$i$={iv:.5f}  ($\Lambda$={na['Lambda']:.0f})")
g_fin = np.mean([g[-1] for _i, _t, g in series])
ax.axhline(g_fin, color='k', ls='--', lw=1.2, label=f'value at $t$=2400 s = {g_fin:.3f}')
ax.set_xlabel(r'$t$ (s)', fontsize=11)
ax.set_ylabel('grading index', fontsize=11)
ax.set_title(r'(c) the slope controls the rate, not the structure\n      ($\phi_s^0$=0.70)', fontsize=11, loc='left')
ax.legend(fontsize=8.5, loc='lower right', framealpha=0.92)
_ejes(ax)
plt.show()
